In [ ]:
import torch

print(torch.cuda.is_available())
torch.cuda.empty_cache()

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.environ['ROBOFLOW_KEY']

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key)
project = rf.workspace("johns-workspace-dsz12").project("bus_passenger")
version = project.version(14)
dataset = version.download("yolo26")
                    

In [ ]:
from ultralytics import YOLO
model = YOLO("runs/detect/SmartByahe/yolo26s_tunedv2_v8/weights/last.pt")
model.train(resume=True)

In [ ]:
from ultralytics import YOLO


model = YOLO("yolo26s.pt")

model.train(
    data="Bus_passenger-14/data.yaml", 
    epochs=700,
    batch=32,
    cfg="runs/detect/tune2/best_hyperparameters.yaml",
    optimizer="AdamW",
    project="SmartByahe",
    name="yolo26s_tunedv2_v8",
    )


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")
model.tune(
    data="Bus_passenger-11/data.yaml", 
    epochs=50,
    iterations=70,
    batch=24,
    optimizer="AdamW",
    plots=True,
    save=False,
    device=0,
    exist_ok=True,
)

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/SmartByahe/yolo26s_tunedv2_v6/weights/best.pt")

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list containing mAP50-95 for each category


In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/SmartByahe/yolo26s_tunedv2_v5/weights/best.pt")

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list containing mAP50-95 for each category


In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/SmartByahe/yolo26s_tunedv2_v7/weights/best.pt")

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list containing mAP50-95 for each category


In [ ]:
from ultralytics import YOLO
model = YOLO("runs/detect/SmartByahe/yolo26s_tunedv2_v5/weights/best.pt")

results = model([
    "dataset/unseen/images/IMG_1706.jpg", 
    "dataset/unseen/images/IMG_1710.jpg",
    "dataset/unseen/images/IMG_1747.jpg",
    "dataset/unseen/images/IMG_1743.jpg"
    ]) 

# Process results list
# for result in results:
#     boxes = result.boxes  # Boxes object for bounding box outputs
#     masks = result.masks  # Masks object for segmentation masks outputs
#     keypoints = result.keypoints  # Keypoints object for pose outputs
#     probs = result.probs  # Probs object for classification outputs
#     obb = result.obb  # Oriented boxes object for OBB outputs
#     result.show()  # display to screen
#     #result.save(filename="result.jpg")  # save to disk

for i, result in enumerate(results):
    result.plot()
    result.save(filename=f"predict_results/result_{i}.jpg")



In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/SmartByahe/yolo26s_tunedv2_v8/weights/best.pt")
source = "dataset/unseen/videos/predict_vid2.mov"
result = model.predict(
    source, 
    save=True, 
    conf=0.5,
    stream=True,
    save_frames=True,
    vid_stride=4,
    project="SmartByahe_Predict",
    name="yolo26s_predict_video",
)

for r in result: 
    pass

# for r in result:
#     boxes = r.boxes  # Boxes object for bounding box outputs
#     masks = r.masks  # Masks object for segmentation masks outputs
#     keypoints = r.keypoints  # Keypoints object for pose outputs
#     probs = r.probs  # Probs object for classification outputs
#     obb = r.obb  # Oriented boxes object for OBB outputs
#     r.save(filename="result.jpg")  # save to disk Oriented boxes object for OBB outputs

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLO26n model
model = YOLO("runs/detect/SmartByahe/yolo26s_tunedv2_v8/weights/best.pt")

# Run inference on the source
results = model(
    source=0, 
    show=True, 
    stream=True,
    )  # generator of Results objects

for result in results:
    boxes = result.boxes
    classes = result.names

In [ ]:

from ultralytics import YOLO

# Load a pretrained YOLO26n model
model = YOLO("runs/detect/SmartByahe/yolo26s_tunedv2_v8/weights/best.pt")

# Run inference on the source
tracker = model.track(
    source=0, 
    tracker="botsort.yaml",
    conf=0.6,
    iou=0.8,
    stream=True,
    show=True,
    persist=True,
    vid_stride=2
    )  # generator of Results 



In [ ]:
import cv2
from ultralytics import solutions

model = "runs/detect/SmartByahe/yolo26s_tunedv2_v8/weights/best.pt"
cap = cv2.VideoCapture(0)
assert cap.isOpened()

#  top-left  top-right  bot-right  bot-left
# [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
region_points = {
    "region-01": [(0, 20), (300, 20), (300, 250), (250, 400), (0, 400)],
    "region-02": [(340, 250), (340, 20), (640, 20), (640, 400), (400, 400)],
}

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
fps = fps if fps > 0 else 30

regionCounter = solutions.RegionCounter(
    show=True,
    region=region_points,
    model=model,
    conf=0.6
)

while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = regionCounter(im0)
    
    print(results)

cap.release()
cv2.destroyAllWindows()